In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

from warnings import filterwarnings
filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense, Input, Dropout, BatchNormalization, Concatenate
from tensorflow.keras.models import Sequential

from tensorflow.keras.optimizers.legacy import Optimizer
from tensorflow.keras.initializers import HeNormal
from tensorflow.keras.regularizers import l2

from tensorflow.keras.optimizers import AdamW
import pickle

from time import perf_counter

2025-04-27 14:12:42.538373: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-27 14:12:42.570779: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-04-27 14:12:42.570864: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-04-27 14:12:42.572072: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-27 14:12:42.577833: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-27 14:12:42.578488: I tensorflow/core/platform/cpu_feature_guard.cc:1

In [2]:
counts = pd.read_csv('/work/counts_pred.csv')

X = pd.read_csv('/work/preprocessed.csv')
X['No Of Withdrawals'] = StandardScaler().fit_transform(counts)
y = pd.read_csv('/work/target.csv')['Total amount Withdrawn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

y_train_norm = (y_train - y_train.min()) / (y_train.max() - y_train.min())
y_test_norm = (y_test - y_test.min()) / (y_test.max() - y_test.min())


In [3]:
# # from keras.saving import register_keras_serializable

# # @keras.utils.register_keras_serializable()
# def cwc_loss(y_true, y_pred):

#     # y_pred: shape (batch, 2) -> [lower, upper]

#     y_lower = y_pred[0]
#     y_upper = y_pred[1]

#     eta=20
#     mu=0.95
#     epsilon = 1e-6  # Small positive constant

#     # Ensure width is positive and stable
#     width = y_upper - y_lower + epsilon

#     # --- PINRW ---
#     R = tf.reduce_max(y_true) - tf.reduce_min(y_true)
#     R = tf.maximum(R, epsilon)  # avoid division by zero
#     pinrw = tf.sqrt(tf.reduce_mean(tf.square(width))) / (R + epsilon)

#     # --- PICP ---
#     in_interval = tf.cast((y_true >= y_lower) & (y_true <= y_upper), tf.float32)
#     picp = tf.reduce_mean(in_interval)

#     # --- Penalty ---
#     gamma = tf.cast(picp < mu, tf.float32)

#     # gamma = tf.sigmoid(100 * (mu - picp))
#     penalty = gamma * tf.exp(eta * (mu - picp))  # capped exponent

#     # --- CWC ---
#     cwc = pinrw * (1.0 + penalty)

#     # Prevent NaNs
#     cwc = tf.where(tf.math.is_nan(cwc), tf.constant(1e6, dtype=cwc.dtype), cwc)
    
#     return cwc


In [4]:
# 6. BUILD MODEL
def build_model(input_shape):
    inputs = Input(shape=(input_shape,))
    x = Dense(64, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.3)(x)
    x = BatchNormalization()(x)

    lower = Dense(1, activation='sigmoid')(x)
    upper = Dense(1, activation='sigmoid')(x)
    return Model(inputs=inputs, outputs=[lower, upper])

# 7. TRAIN MODEL

# Targets are synthetic intervals for initial training (for supervised signal)
lower_bound_init = y_train_norm - 0.15
upper_bound_init = y_train_norm + 0.15

lower_bound_init = np.clip(lower_bound_init, 0, 1)
upper_bound_init = np.clip(upper_bound_init, 0, 1)


In [5]:
t1 = perf_counter()

model = build_model(X_train.shape[1])
model.compile(optimizer='adam', loss='mse')  # Using dummy loss for interval fitting

callback = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='loss', factor=0.5, patience=5, min_lr=1e-6)

early_stop = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=20, restore_best_weights=True)


# 4. Build and compile model
model.fit(X_train, [lower_bound_init, upper_bound_init], epochs=200, batch_size=32, verbose=1, callbacks=[callback, early_stop])


t2 = perf_counter()
print(f'Time taken: {t2-t1: .2f} seconds')


Epoch 1/200
317/317 [==============================] - 1s 1ms/step - loss: 0.0876 - dense_3_loss: 0.0487 - dense_4_loss: 0.0389 - lr: 0.0010
Epoch 2/200
317/317 [==============================] - 0s 1ms/step - loss: 0.0267 - dense_3_loss: 0.0126 - dense_4_loss: 0.0141 - lr: 0.0010
Epoch 3/200
317/317 [==============================] - 0s 1ms/step - loss: 0.0178 - dense_3_loss: 0.0085 - dense_4_loss: 0.0093 - lr: 0.0010
Epoch 4/200
317/317 [==============================] - 0s 1ms/step - loss: 0.0146 - dense_3_loss: 0.0070 - dense_4_loss: 0.0076 - lr: 0.0010
Epoch 5/200
317/317 [==============================] - 0s 1ms/step - loss: 0.0133 - dense_3_loss: 0.0063 - dense_4_loss: 0.0069 - lr: 0.0010
Epoch 6/200
317/317 [==============================] - 0s 1ms/step - loss: 0.0133 - dense_3_loss: 0.0064 - dense_4_loss: 0.0069 - lr: 0.0010
Epoch 7/200
317/317 [==============================] - 0s 2ms/step - loss: 0.0123 - dense_3_loss: 0.0059 - dense_4_loss: 0.0064 - lr: 0.0010
Epoch 8/200
3

In [6]:
# 8. PREDICT INTERVALS
# y_lower_pred, y_upper_pred = model.predict(X_test)
# y_lower_pred = y_lower_pred.flatten()
# y_upper_pred = y_upper_pred.flatten()
# y_lower_pred = np.minimum(y_lower_pred, y_upper_pred)
# y_upper_pred = np.maximum(y_lower_pred, y_upper_pred)

# # 9. CLIP OUTPUTS
# y_lower_pred = np.clip(y_lower_pred, 0, 1)
# y_upper_pred = np.clip(y_upper_pred, 0, 1)


In [7]:

pred = model.predict(X_test)
pred = np.array(pred)

lb_amt = pred[0, :].flatten()
ub_amt = pred[1, :].flatten()


lb_amt = (lb_amt * (y_test.max() - y_test.min()) + y_test.min()).astype('int')
ub_amt = (ub_amt * (y_test.max() - y_test.min()) + y_test.min()).astype('int')

lb_amt = lb_amt.astype('int')
ub_amt = ub_amt.astype('int')


80/80 [==============================] - 0s 817us/step


In [8]:

below_lb = (lb_amt >= y_test).sum() / len(y_test)
print('Points Below Lower Bound:', below_lb)

below_ub = (ub_amt >= y_test).sum() / len(y_test)
print('Points Below Upper Bound:', below_ub)

Points Below Lower Bound: 0.020118343195266272
Points Below Upper Bound: 0.9743589743589743


In [9]:
def compute_picp(y_lower, y_upper, y_true):
    return np.mean(np.logical_and(y_true >= y_lower, y_true <= y_upper))

def compute_pinrw(y_lower, y_upper, y_true):
    R = np.max(y_true) - np.min(y_true)
    width = np.sqrt(np.mean((y_upper - y_lower) ** 2))
    return width / R

def cwc_loss_numpy(y_lower, y_upper, y_true, gamma=50, eta=50, mu=0.95):
    picp = compute_picp(y_lower, y_upper, y_true)
    pinrw = compute_pinrw(y_lower, y_upper, y_true)
    penalty = np.exp(eta * (mu - picp))
    cwc = pinrw * (1 + gamma * penalty)
    mpiw = np.mean(y_upper - y_lower)

    return cwc, picp, mpiw, pinrw

In [13]:

cwc, picp, mpiw, pinrw = cwc_loss_numpy(lb_amt, ub_amt, y_test, gamma=50, eta=50, mu=0.95)
outcome = (y_test >= lb_amt) & (y_test <= ub_amt)

result = pd.DataFrame({'Original': y_test, 'Predicted': (lb_amt + ub_amt) / 2,
                        'Lower Bound': lb_amt, 'Upper Bound': ub_amt,
                        'Outcome': outcome})


result.to_csv(f'Results_PINRW_amt.csv', index=False)


In [16]:
print('PICP: ', picp)
print('PINRW: ', pinrw)
print('CWC: ', cwc)
print('MPIW: ', mpiw)

PICP:  0.954240631163708
PINRW:  0.29220914054043545
CWC:  12.111180099551195
MPIW:  354214.28205128206


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=6c5dcd7e-ee44-459a-9dd6-5c39b2404cb3' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>